In [4]:
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import fbeta_score,classification_report, average_precision_score

In [5]:
# Define features and target variable
features = [
    # encoded station/service
    'StationCode_TE',
    'Service:Type_Intercity',
    'Service:Type_Intercity direct',
    'Service:Type_Sprinter',
    # temporal
    'Hour_sin',
    'Hour_cos',
    'DayOfWeek_sin',
    'DayOfWeek_cos',
    'Month_sin',
    'Month_cos',
    'IsWeekend',
    'RushHour',
    # operational context
    'StationTraffic',
    'Stop:Platform change',
    'arrival_scheduled',
    'departure_scheduled',
    # weather
    'Wind Direction',
    'Hourly Mean Wind Speed',
    'Wind Speed last 10 Minutes',
    'Max Wind Speed',
    'Temperature',
    'Dew Point temperature',
    'Sunshine Duration',
    'Global Radiation',
    'Precipitation Duration',
    'Precipitation Amount',
    'Air Pressure',
    'Horizontal Visibility',
    'Cloud Cover',
    'Humidity',
    'Fog',
    'Rainfall',
    'Snowfall',
    'Thunder',
    'Hail',
]

target = "is_cancelled"

In [ ]:
# Prepare for chunked reading
usecols = features + [target]
chunk_size = 1_000_000
sample_size = 1_000_000
random_state = 42

dtype_map = {col: "float32" for col in features}
dtype_map[target] = "int8"


# Read CSV in chunks
def chunk_reader(file_path):
    for chunk in pd.read_csv(
        file_path,
        usecols=usecols,
        dtype=dtype_map,
        chunksize=chunk_size
    ):
        chunk = chunk.reindex(columns=usecols, fill_value=0)

        X_chunk = chunk[features]
        y_chunk = chunk[target]

        yield X_chunk, y_chunk


# Count classes
neg_total = 0
pos_total = 0

for X_chunk, y_chunk in chunk_reader("train_data.csv"):
    neg_total += (y_chunk == 0).sum()
    pos_total += (y_chunk == 1).sum()

total_rows = neg_total + pos_total

print(f"Train not cancelled: {neg_total:,}")
print(f"Train cancelled: {pos_total:,}")
print(f"Train total:     {total_rows:,}")

# Decide stratified sample sizes
neg_sample_size = int(sample_size * neg_total / total_rows)
pos_sample_size = sample_size - neg_sample_size

print(f"Sampling not cancelled: {neg_sample_size:,}")
print(f"Sampling cancelled: {pos_sample_size:,}")

# stratified sample from chunks
rng = np.random.default_rng(random_state)

X_neg_parts = []
y_neg_parts = []
X_pos_parts = []
y_pos_parts = []

neg_seen = 0
pos_seen = 0

for X_chunk, y_chunk in chunk_reader("train_data.csv"):

    neg_mask = y_chunk == 0
    pos_mask = y_chunk == 1

    X_neg = X_chunk.loc[neg_mask]
    y_neg = y_chunk.loc[neg_mask]

    X_pos = X_chunk.loc[pos_mask]
    y_pos = y_chunk.loc[pos_mask]

    # sample fraction based on remaining needed / remaining available
    neg_remaining_needed = neg_sample_size - sum(len(p) for p in y_neg_parts)
    pos_remaining_needed = pos_sample_size - sum(len(p) for p in y_pos_parts)

    neg_remaining_total = neg_total - neg_seen
    pos_remaining_total = pos_total - pos_seen

    if neg_remaining_needed > 0 and len(X_neg) > 0:
        p_neg = min(1.0, neg_remaining_needed / max(neg_remaining_total, 1))
        keep_neg = rng.random(len(X_neg)) < p_neg

        X_neg_parts.append(X_neg.loc[keep_neg])
        y_neg_parts.append(y_neg.loc[keep_neg])

    if pos_remaining_needed > 0 and len(X_pos) > 0:
        p_pos = min(1.0, pos_remaining_needed / max(pos_remaining_total, 1))
        keep_pos = rng.random(len(X_pos)) < p_pos

        X_pos_parts.append(X_pos.loc[keep_pos])
        y_pos_parts.append(y_pos.loc[keep_pos])

    neg_seen += len(X_neg)
    pos_seen += len(X_pos)


X_train_sample = pd.concat(X_neg_parts + X_pos_parts, axis=0)
y_train_sample = pd.concat(y_neg_parts + y_pos_parts, axis=0)

# Shuffle final sample
shuffle_idx = rng.permutation(len(X_train_sample))
X_train_sample = X_train_sample.iloc[shuffle_idx].to_numpy(dtype=np.float32, copy=False)
y_train_sample = y_train_sample.iloc[shuffle_idx].to_numpy(copy=False)

print(f"Final XGBoost sample size: {len(y_train_sample):,}")


# Handle class imbalance
neg_count = (y_train_sample == 0).sum()
pos_count = (y_train_sample == 1).sum()

scale_pos_weight = neg_count / max(pos_count, 1)

print(f"scale_pos_weight: {scale_pos_weight:.4f}")


# Train XGBoost
xgb = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.01,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric="auc",
    random_state=random_state,
    n_jobs=-1,
    tree_method="hist"
)

xgb.fit(X_train_sample, y_train_sample)

print(f"XGBoost trained on {len(y_train_sample):,} stratified samples")

Train negatives: 43,903,090
Train positives: 4,479,366
Train total:     48,382,456
Sampling negatives: 907,417
Sampling positives: 92,583
Final XGBoost sample size: 999,797
scale_pos_weight: 9.8067
XGBoost trained on 999,797 stratified samples


In [7]:
# Evaluate CSV split in chunks
def evaluate_split_xgb(name, file_path):
    y_true_parts = []
    y_pred_parts = []
    y_prob_parts = []

    for X_chunk, y_chunk in chunk_reader(file_path):
        X_chunk_np = X_chunk.to_numpy(dtype=np.float32, copy=False)

        y_true_parts.append(y_chunk.to_numpy(copy=False))
        y_pred_parts.append(xgb.predict(X_chunk_np))
        y_prob_parts.append(xgb.predict_proba(X_chunk_np)[:, 1])

    y_true_all = np.concatenate(y_true_parts)
    y_pred_all = np.concatenate(y_pred_parts)
    y_prob_all = np.concatenate(y_prob_parts)

    f2 = fbeta_score(y_true_all, y_pred_all, beta=2)

    print(f"\n{name}")
    print("F2 Score:", round(f2, 4))
    print(classification_report(y_true_all, y_pred_all, digits=3, zero_division=0))
    print("PR-AUC:", round(average_precision_score(y_true_all, y_prob_all), 4))


evaluate_split_xgb("Validation", "val_data.csv")
evaluate_split_xgb("Test", "test_data.csv")


Validation
F2 Score: 0.3653
              precision    recall  f1-score   support

           0      0.931     0.670     0.779   9333880
           1      0.156     0.550     0.243   1033789

    accuracy                          0.658  10367669
   macro avg      0.543     0.610     0.511  10367669
weighted avg      0.853     0.658     0.726  10367669

PR-AUC: 0.167

Test
F2 Score: 0.3917
              precision    recall  f1-score   support

           0      0.913     0.664     0.769   9105889
           1      0.184     0.546     0.275   1261781

    accuracy                          0.649  10367670
   macro avg      0.549     0.605     0.522  10367670
weighted avg      0.825     0.649     0.709  10367670

PR-AUC: 0.2027


In [ ]:
# Feature importances

importances = pd.Series(
    xgb.feature_importances_,
    index=features
).sort_values(ascending=False)

print("\nTop 20 Feature Importances:")
print(importances.head(20))


Top 20 Feature Importances:
Service:Type_Sprinter            0.492887
Service:Type_Intercity           0.237411
StationCode_TE                   0.025101
IsWeekend                        0.019287
Stop:Platform change             0.018623
Hour_cos                         0.016069
Month_sin                        0.014215
DayOfWeek_sin                    0.011795
departure_scheduled              0.011593
StationTraffic                   0.011477
arrival_scheduled                0.011034
Hourly Mean Wind Speed           0.010982
Service:Type_Intercity direct    0.010972
Max Wind Speed                   0.010848
Air Pressure                     0.010121
Wind Speed last 10 Minutes       0.009897
Rainfall                         0.009673
Dew Point temperature            0.007844
Temperature                      0.007732
DayOfWeek_cos                    0.006702
dtype: float32
